# Module 6 Learning Activity: Decision Tree and Naïve Bayes Algorithms

This notebook is a beginner-friendly practical tutorial that supports the Module 6 learning activities.

The official activity asks you to think through a small decision tree exercise and discuss missing values. This notebook shows the same ideas in code using a simple student-grade dataset:

- create a small synthetic dataset;
- convert categorical text values into numeric values;
- build a **Decision Tree** classifier;
- briefly compare with a **Naïve Bayes** classifier;
- evaluate model predictions; and
- discuss how to handle missing values in an important attribute.

The code is intentionally kept simple. Focus on the workflow and the meaning of each function rather than trying to memorise every line.

## 0. Optional setup note

If you are using **Google Colab**, PySpark may not already be installed. If Spark is missing, run the following command in a separate cell:
If you are using a local Anaconda/Jupyter environment where PySpark is already installed, you can skip this.

In [2]:
!pip install pyspark

## 1. Create a small dataset

To make the machine learning workflow easy to understand, we will create a synthetic dataset named `students_grades.csv`.

The dataset has **input features** such as age, gender, study hours, participation, major, school type, and region. The output variable is `Grade`, which has two possible classes: `Pass` and `Fail`.

The first cell below creates a completely random dataset. It is useful for understanding data structure, but a model may not learn much from completely random data.

In [3]:
# Create a completely random dataset
import pandas as pd
import random

data = {
    "Age": [random.randint(18, 30) for _ in range(1000)],
    "Gender": [random.choice(["Male", "Female"]) for _ in range(1000)],
    "StudyHours": [random.randint(5, 25) for _ in range(1000)],
    "Participation": [random.randint(1, 10) for _ in range(1000)],
    "Major": [random.choice(["Computer Science", "Biology", "Business", "Literature", "Physics"]) for _ in range(1000)],
    "TypeOfSchool": [random.choice(["Public", "Private", "Online"]) for _ in range(1000)],
    "Region": [random.choice(["North", "South", "East", "West", "Central"]) for _ in range(1000)],
    "Grade": [random.choice(["Pass", "Fail"]) for _ in range(1000)]
}

df = pd.DataFrame(data)
df.to_csv("students_grades.csv", index=False)

The next cell creates a dataset with a stronger relationship between inputs and output.

For example, students with higher study hours and higher participation are more likely to pass. This makes the classification task more meaningful because the model can learn patterns from the data.

When you run all cells, this second dataset will overwrite the previous `students_grades.csv` file. For teaching purposes, this is fine because we want a dataset with learnable patterns.

In [4]:
# Create a dataset with stronger relationship between Inputs and Outputs
import pandas as pd
import random

def determine_grade(study_hours, participation, school_type):
    # Base probability of passing
    base_prob = 0.4  # Adjusted down to allow for larger swings based on criteria

    # Increase the probability based on study hours
    if study_hours > 20:
        base_prob += 0.4
    elif study_hours > 15:
        base_prob += 0.3
    elif study_hours > 10:
        base_prob += 0.2
    elif study_hours <= 10:
        base_prob -= 0.1

    # Increase the probability based on participation
    if participation > 8:
        base_prob += 0.3
    elif participation > 6:
        base_prob += 0.2
    elif participation <= 5:
        base_prob -= 0.2

    # Adjust the probability based on school type
    if school_type == "Private":
        base_prob += 0.2
    elif school_type == "Online":
        base_prob -= 0.2

    # Final decision
    return "Pass" if random.random() < base_prob else "Fail"

data = {
    "Age": [random.randint(18, 30) for _ in range(1000)],
    "Gender": [random.choice(["Male", "Female"]) for _ in range(1000)],
    "StudyHours": [random.randint(5, 25) for _ in range(1000)],
    "Participation": [random.randint(1, 10) for _ in range(1000)],
    "Major": [random.choice(["Computer Science", "Biology", "Business", "Literature", "Physics"]) for _ in range(1000)],
    "TypeOfSchool": [random.choice(["Public", "Private", "Online"]) for _ in range(1000)],
    "Region": [random.choice(["North", "South", "East", "West", "Central"]) for _ in range(1000)],
}
data["Grade"] = [determine_grade(data["StudyHours"][i], data["Participation"][i], data["TypeOfSchool"][i]) for i in range(1000)]

df = pd.DataFrame(data)
df.to_csv("students_grades.csv", index=False)

## 2. Import packages

The next cell imports the tools we need.

Key packages:

- `SparkSession`: starts a Spark application.
- `StringIndexer`: converts text categories into numbers.
- `VectorAssembler`: combines several input columns into one `features` vector.
- `DecisionTreeClassifier`: builds a decision tree classification model.
- `NaiveBayes`: builds a Naïve Bayes classification model.
- `Pipeline`: runs multiple preparation steps in order.
- `BinaryClassificationEvaluator`: calculates AUC for binary classification.
- `MulticlassMetrics`: helps create a confusion matrix.
- `classification_report`: gives precision, recall and F1-score.

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, NaiveBayes
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from sklearn.metrics import classification_report

## 3. Start Spark

`SparkSession` is the entry point for working with Spark DataFrames.

- `SparkSession.builder` starts the setup process.
- `.appName("SupervisedLearning")` gives a name to this Spark application.
- `.getOrCreate()` creates a new Spark session or reuses an existing one.

In [6]:
# Initialize Spark Session
spark = SparkSession.builder.appName("SupervisedLearning").getOrCreate()

## 4. Load the CSV file into a Spark DataFrame

The command below reads the CSV file we created earlier.

Arguments:

- `header=True`: the first row contains column names.
- `inferSchema=True`: Spark tries to detect column types automatically, such as integer or string.

After loading, `df` is a Spark DataFrame.

In [7]:
# Load the dataset
df = spark.read.csv('students_grades.csv', header=True, inferSchema=True)

## 5. Handle missing data using simple deletion

`dropna()` removes rows that contain missing values.

This is the simplest method and is easy for beginners. However, in real projects, deleting rows may not always be the best approach, especially if many values are missing or if the missing values occur in an important attribute.

Later in this notebook, we will also discuss a simple replacement method.

In [8]:
# Handle missing data by deletion
df = df.dropna()

## 6. Inspect the schema

`printSchema()` shows each column name and its data type.

This is important because machine learning models usually require numeric input. In this dataset, some columns are numeric already, while others are text categories.

In [9]:
# Showing the type of each column
df.printSchema()

root
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- StudyHours: integer (nullable = true)
 |-- Participation: integer (nullable = true)
 |-- Major: string (nullable = true)
 |-- TypeOfSchool: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Grade: string (nullable = true)




In machine learning, the categorial data are generall encoded before running a ML algorithm.

#### What is Categorical Data?

- Categorical data are variables that contain label values rather than numeric values.
- The number of possible values is often limited to a fixed set.
- Categorical variables are often called **Nominal**.

Some examples include:

A “pet” variable with the values: “dog” and “cat“.
A “color” variable with the values: “red“, “green” and “blue“.
A “place” variable with the values: “first”, “second” and “third“.

#### What is the Problem with Categorical Data?
- Some algorithms can work with categorical data directly.
- Many machine learning algorithms cannot operate on label data directly. They require all input variables and output variables to be numeric.

#### Solution: Convert Categorical Data to Numerical Data:



## 7. List the categorical columns

These columns contain text labels. Before using them in a Spark ML model, we need to convert them into numeric columns.

Notice that `Grade` is included because it is the **output label**. The model needs the label to be numeric as well.

In [10]:
# Create a list including all categorical columns of INPUTS
categorical_cols = ['Gender', 'Major', 'TypeOfSchool', 'Region', 'Grade']

#### StringIndexer:

The StringIndexer is a vital PySpark feature that helps convert categorical string columns in a DataFrame into numerical indices.


#### Pipeline:
Pipeline is a tool from the PySpark ML library that allows for the chaining and structuring of multiple stages of data processing and/or modeling steps.

`stages=indexers` means that the pipeline is being constructed with a series of stages that are represented by the indexers list. Each stage in indexers represents a StringIndexer transformation, which is used to convert categorical string columns into numeric indices.

The code below creates one `StringIndexer` for each categorical column.

Important arguments:

- `inputCol=col`: the original text column.
- `outputCol=col + "Numeric"`: the new numeric column name.
- `.fit(df)`: learns the mapping from text labels to numbers.
- `.transform(df)`: applies the learned mapping to the dataset.

For example, `Gender` becomes `GenderNumeric`, and `Grade` becomes `GradeNumeric`.

In [11]:
indexers = [StringIndexer(inputCol=col, outputCol=col + "Numeric").fit(df) for col in categorical_cols]

pipeline = Pipeline(stages=indexers)
df_encoded = pipeline.fit(df).transform(df)

`toPandas()` converts the Spark DataFrame into a Pandas DataFrame for display.

This is useful for small teaching datasets. For very large datasets, avoid converting the full Spark DataFrame to Pandas because it may use too much memory.

In [12]:
# Show the dataset
df_encoded.toPandas()

,Age,Gender,StudyHours,Participation,Major,TypeOfSchool,Region,Grade,GenderNumeric,MajorNumeric,TypeOfSchoolNumeric,RegionNumeric,GradeNumeric
0,30,Male,22,6,Business,Public,West,Pass,1.0,4.0,1.0,2.0,0.0
1,28,Female,14,3,Computer Science,Online,North,Pass,0.0,2.0,0.0,0.0,0.0
2,20,Male,14,7,Business,Public,West,Fail,1.0,4.0,1.0,2.0,1.0
3,23,Female,6,10,Biology,Public,South,Pass,0.0,3.0,1.0,3.0,0.0
4,20,Female,21,8,Literature,Private,North,Pass,0.0,0.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,Female,11,3,Biology,Private,North,Pass,0.0,3.0,2.0,0.0,0.0
996,26,Male,11,1,Business,Private,Central,Pass,1.0,4.0,2.0,1.0,0.0
997,25,Male,15,10,Literature,Online,East,Fail,1.0,0.0,0.0,4.0,1.0
998,29,Female,14,9,Literature,Public,West,Pass,0.0,0.0,1.0,2.0,0.0


### VectorAssembler

VectorAssembler is a transformer in PySpark's MLlib that combines a given list of columns into a **single vector** column. It is commonly used in the preprocessing stages of a machine learning pipeline to bring together features into one aggregate column, which is often a requirement for ML algorithms in Spark.

## 8. Assemble input features into one vector

Spark ML models expect the input variables to be stored in a single column, usually called `features`.

`VectorAssembler` combines the selected input columns into this one vector column.

Arguments:

- `inputCols=[...]`: all input variables used by the model.
- `outputCol='features'`: the name of the new vector column.

In [13]:
# Define feature columns and assemble them as a vector
assembler = VectorAssembler(
    inputCols=['Age', 'GenderNumeric', 'StudyHours', 'Participation', 'MajorNumeric', 'TypeOfSchoolNumeric', 'RegionNumeric'],
    outputCol='features')

df_assembled = assembler.transform(df_encoded)

Now, all Inputs(features) have been assembled into a single vector, titled as 'features'.

In [14]:
df_assembled.toPandas()

,Age,Gender,StudyHours,Participation,Major,TypeOfSchool,Region,Grade,GenderNumeric,MajorNumeric,TypeOfSchoolNumeric,RegionNumeric,GradeNumeric,features
0,30,Male,22,6,Business,Public,West,Pass,1.0,4.0,1.0,2.0,0.0,"[30.0, 1.0, 22.0, 6.0, 4.0, 1.0, 2.0]"
1,28,Female,14,3,Computer Science,Online,North,Pass,0.0,2.0,0.0,0.0,0.0,"[28.0, 0.0, 14.0, 3.0, 2.0, 0.0, 0.0]"
2,20,Male,14,7,Business,Public,West,Fail,1.0,4.0,1.0,2.0,1.0,"[20.0, 1.0, 14.0, 7.0, 4.0, 1.0, 2.0]"
3,23,Female,6,10,Biology,Public,South,Pass,0.0,3.0,1.0,3.0,0.0,"[23.0, 0.0, 6.0, 10.0, 3.0, 1.0, 3.0]"
4,20,Female,21,8,Literature,Private,North,Pass,0.0,0.0,2.0,0.0,0.0,"[20.0, 0.0, 21.0, 8.0, 0.0, 2.0, 0.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,Female,11,3,Biology,Private,North,Pass,0.0,3.0,2.0,0.0,0.0,"[27.0, 0.0, 11.0, 3.0, 3.0, 2.0, 0.0]"
996,26,Male,11,1,Business,Private,Central,Pass,1.0,4.0,2.0,1.0,0.0,"[26.0, 1.0, 11.0, 1.0, 4.0, 2.0, 1.0]"
997,25,Male,15,10,Literature,Online,East,Fail,1.0,0.0,0.0,4.0,1.0,"[25.0, 1.0, 15.0, 10.0, 0.0, 0.0, 4.0]"
998,29,Female,14,9,Literature,Public,West,Pass,0.0,0.0,1.0,2.0,0.0,"[29.0, 0.0, 14.0, 9.0, 0.0, 1.0, 2.0]"


From this point forward, we just need two columns:
1. **features** which includes all Inputs
2. **GradeNumeric** which is the Output of the model

In [15]:
# Filtering the Input and Output columns into a new dataframe
df_assembled_filtered = df_assembled.select("features", "GradeNumeric")

In [16]:
df_assembled_filtered.toPandas()

,features,GradeNumeric
0,"[30.0, 1.0, 22.0, 6.0, 4.0, 1.0, 2.0]",0.0
1,"[28.0, 0.0, 14.0, 3.0, 2.0, 0.0, 0.0]",0.0
2,"[20.0, 1.0, 14.0, 7.0, 4.0, 1.0, 2.0]",1.0
3,"[23.0, 0.0, 6.0, 10.0, 3.0, 1.0, 3.0]",0.0
4,"[20.0, 0.0, 21.0, 8.0, 0.0, 2.0, 0.0]",0.0
...,...,...
995,"[27.0, 0.0, 11.0, 3.0, 3.0, 2.0, 0.0]",0.0
996,"[26.0, 1.0, 11.0, 1.0, 4.0, 2.0, 1.0]",0.0
997,"[25.0, 1.0, 15.0, 10.0, 0.0, 0.0, 4.0]",1.0
998,"[29.0, 0.0, 14.0, 9.0, 0.0, 1.0, 2.0]",0.0


## 9. Build the Decision Tree model

A decision tree learns a set of if/else-style rules from the training data.

For example, the tree may learn that students with high study hours and high participation are more likely to pass.

We first split the data into two parts:

- **Training data**: used to build the model.
- **Testing data**: used to evaluate the model on data it has not seen before.

`randomSplit([0.8, 0.2])` means approximately 80% of the data is used for training and 20% for testing.

In [17]:
# Train-Test split
train_data, test_data = df_assembled_filtered.randomSplit([0.8, 0.2])

Now we create and train the decision tree.

Arguments:

- `featuresCol='features'`: tells Spark where the input variables are stored.
- `labelCol='GradeNumeric'`: tells Spark which column is the target/output.
- `.fit(train_data)`: trains the model using the training data.

In [18]:
# Train a Decision Tree model
dtc = DecisionTreeClassifier(featuresCol='features', labelCol="GradeNumeric")
model = dtc.fit(train_data)

### Prediction using the Trained Model

## 10. Make predictions

`model.transform(test_data)` applies the trained model to the testing dataset.

The result contains several useful columns:

- `features`: the input values.
- `GradeNumeric`: the actual class.
- `rawPrediction`: model score for each class.
- `probability`: estimated probability for each class.
- `prediction`: the model's predicted class.

In [19]:
# Predictions using test_data
predictions = model.transform(test_data)

In [20]:
# "Raw prediction" for each possible label. The meaning of a "raw" prediction may vary between algorithms, but it intuitively gives a measure of confidence in each possible label (where larger = more confident).
predictions.toPandas()

,features,GradeNumeric,rawPrediction,probability,prediction
0,"[18.0, 0.0, 7.0, 10.0, 3.0, 0.0, 1.0]",1.0,"[9.0, 19.0]","[0.32142857142857145, 0.6785714285714286]",1.0
1,"[18.0, 0.0, 18.0, 5.0, 0.0, 1.0, 3.0]",0.0,"[49.0, 37.0]","[0.5697674418604651, 0.43023255813953487]",0.0
2,"[18.0, 0.0, 19.0, 2.0, 0.0, 0.0, 2.0]",0.0,"[7.0, 22.0]","[0.2413793103448276, 0.7586206896551724]",1.0
3,"[18.0, 1.0, 6.0, 6.0, 4.0, 1.0, 1.0]",0.0,"[16.0, 142.0]","[0.10126582278481013, 0.8987341772151899]",1.0
4,"[18.0, 1.0, 6.0, 8.0, 4.0, 2.0, 3.0]",0.0,"[29.0, 6.0]","[0.8285714285714286, 0.17142857142857143]",0.0
...,...,...,...,...,...
199,"[30.0, 0.0, 21.0, 2.0, 2.0, 0.0, 2.0]",0.0,"[7.0, 22.0]","[0.2413793103448276, 0.7586206896551724]",1.0
200,"[30.0, 1.0, 16.0, 6.0, 2.0, 0.0, 4.0]",1.0,"[11.0, 44.0]","[0.2, 0.8]",1.0
201,"[30.0, 1.0, 17.0, 7.0, 3.0, 1.0, 0.0]",0.0,"[144.0, 4.0]","[0.972972972972973, 0.02702702702702703]",0.0
202,"[30.0, 1.0, 17.0, 10.0, 1.0, 0.0, 0.0]",1.0,"[37.0, 5.0]","[0.8809523809523809, 0.11904761904761904]",0.0


## 11. Inspect the decision tree rules

`toDebugString` prints the learned decision rules.

This is useful because decision trees are relatively interpretable. You can read the splits and describe the model in words, which directly connects to the discussion forum activity.

In [22]:
# Print Decision Tree rules
print(model.toDebugString)


DecisionTreeClassificationModel: uid=DecisionTreeClassifier_3d44b65dcb97, depth=5, numNodes=29, numClasses=2, numFeatures=7
  If (feature 3 <= 6.5)
   If (feature 2 <= 10.5)
    Predict: 1.0
   Else (feature 2 > 10.5)
    If (feature 5 in {1.0,2.0})
     If (feature 5 in {2.0})
      Predict: 0.0
     Else (feature 5 not in {2.0})
      If (feature 3 <= 1.5)
       Predict: 1.0
      Else (feature 3 > 1.5)
       Predict: 0.0
    Else (feature 5 not in {1.0,2.0})
     If (feature 2 <= 16.5)
      Predict: 1.0
     Else (feature 2 > 16.5)
      If (feature 6 in {0.0,3.0,4.0})
       Predict: 0.0
      Else (feature 6 not in {0.0,3.0,4.0})
       Predict: 1.0
  Else (feature 3 > 6.5)
   If (feature 2 <= 10.5)
    If (feature 5 in {1.0,2.0})
     If (feature 2 <= 9.5)
      If (feature 4 in {0.0,1.0,3.0,4.0})
       Predict: 0.0
      Else (feature 4 not in {0.0,1.0,3.0,4.0})
       Predict: 1.0
     Else (feature 2 > 9.5)
      Predict: 1.0
    Else (feature 5 not in {1.0,2.0})
     Pred

## 12. Feature importance: which input matters most?

Decision trees can estimate how important each input feature was for making predictions.

This connects to Learning Activity 2, where you need to identify the most important attribute and discuss how to handle missing values in that attribute.

In [23]:
feature_names = ['Age', 'GenderNumeric', 'StudyHours', 'Participation', 'MajorNumeric', 'TypeOfSchoolNumeric', 'RegionNumeric']
importances = model.featureImportances.toArray()

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df

,Feature,Importance
3,Participation,0.443613
2,StudyHours,0.344325
5,TypeOfSchoolNumeric,0.155902
6,RegionNumeric,0.033600
4,MajorNumeric,0.022560
1,GenderNumeric,0.000000
0,Age,0.000000


Interpretation guide:

- A larger importance value means the decision tree used that feature more often or more effectively.
- If the top feature has many missing values in a real dataset, you should not simply ignore the issue.
- You need to decide whether to remove rows, replace missing values, or collect better data.

### Evaluate the performance of a binary classification model

**BinaryClassificationEvaluator:** This is an evaluator for binary classification, which expects two input columns: **raw prediction** and **label**.

Parameters:

`rawPredictionCol="rawPrediction"`: This parameter tells the evaluator to expect the column named "rawPrediction" in the dataset (typically predictions in this context) to hold the raw prediction values from the model.
`labelCol="GradeNumeric"`: This parameter tells the evaluator that the true labels for the binary classification task can be found in the "GradeNumeric" column of the dataset.
evaluate() Method:

`evaluator.evaluate(predictions)`: This is where the actual evaluation happens. The evaluate() method computes the metric (Area Under ROC, by default) for the predictions dataset using the true labels and raw predictions.

**Area Under ROC:**

The code calculates the Area Under the Receiver Operating Characteristic (ROC) curve, which is a metric used to evaluate the performance of binary classification models. The value of Area Under ROC (often abbreviated as AUC) ranges between 0 and 1. A value of 0.5 indicates no discriminative power (i.e., the model is as good as random guessing), while a value of 1.0 indicates perfect classification. A higher AUC indicates a better model.

In [24]:
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="GradeNumeric")
area_under_roc = evaluator.evaluate(predictions)
print("Area Under ROC:", area_under_roc)

Area Under ROC: 0.8267692922830867


When dealing with Spark's Machine Learning Library (MLlib), often one needs to evaluate the performance of a model, especially for classification tasks. In order to do that, you often use evaluators that require the prediction and actual label in a specific format.

Convert 'predictions' DataFrame to an **Resilient Distributed Dataset(RDD)** of (prediction, label) tuples" means that you need to transform the DataFrame (predictions) which contains predicted and actual values into a Resilient Distributed Dataset (RDD) that consists of tuples. Each tuple in this RDD contains two elements: the **predicted value** (often the first element) and the **actual label** (often the second element).

Each tuple in this RDD contains two elements: the predicted value (often the first element) and the actual label (often the second element).


In [25]:
#  Convert 'predictions' DataFrame to an RDD of (prediction, label) tuples

prediction_and_label = predictions.select("prediction", "GradeNumeric").rdd.map(lambda row: (float(row["prediction"]), float(row["GradeNumeric"])))
prediction_and_label

PythonRDD[142] at RDD at PythonRDD.scala:56

The RDD below contains pairs in the format:

```text
(predicted label, actual label)
```

This is the format required by `MulticlassMetrics` for the confusion matrix.

In [26]:
# Using 'collect' to show the content of a RDD
for pred, label in prediction_and_label.collect():
    print(f"Prediction: {pred}, Actual Label: {label}")

Prediction: 1.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 1.0, Actual Label: 1.0
Prediction: 0.0, Actual Label: 0.0
Prediction: 0.0, Act

### Confusion Matrix

Where:

- **TN (True Negative):** The number of actual negatives (0s) that were correctly predicted as negatives by the model.
- **FP (False Positive):** The number of actual negatives (0s) that were incorrectly predicted as positives (1s) by the model.
- **FN (False Negative):** The number of actual positives (1s) that were incorrectly predicted as negatives (0s) by the model.
- **TP (True Positive):** The number of actual positives (1s) that were correctly predicted as positives by the model.


###### Interpretation:

**High values of TP and TN, along with low values of FP and FN, generally indicate a good model.**

```text
          Predicted:
          0     |   1
-----------------------
Actual: 0 |  TN    |   FP
-----------------------
Actual: 1 |  FN    |   TP
```

In [27]:
# Create a MulticlassMetrics object to develop the Confusion Matrix
metrics = MulticlassMetrics(prediction_and_label)
confusion_matrix = metrics.confusionMatrix()

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [28]:
# Step 17:Print the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix)

Confusion Matrix:
DenseMatrix([[100.,  22.],
             [ 33.,  49.]])


### Using Scikit-learn package to get a detailed classification report

In [29]:
# Convert 'predictions' DataFrame to a Pandas DataFrame
predictions_pd = predictions.select("prediction", "GradeNumeric").toPandas()

In [30]:
# Step 19: Calculate classification report
report = classification_report(predictions_pd["GradeNumeric"], predictions_pd["prediction"])
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.75      0.82      0.78       122
         1.0       0.69      0.60      0.64        82

    accuracy                           0.73       204
   macro avg       0.72      0.71      0.71       204
weighted avg       0.73      0.73      0.73       204



## 13. Optional: Simple Naïve Bayes model

Naïve Bayes is another supervised classification algorithm. It is based on probability.

In simple terms, it estimates how likely each class is given the observed feature values.

Important teaching note: PySpark's `NaiveBayes` is commonly used with non-negative feature values, especially count-like features. Our synthetic dataset is not a perfect Naïve Bayes use case, but it is still useful as a beginner demonstration of training and prediction workflow.

In [31]:
# Train a simple Naïve Bayes model
nb = NaiveBayes(featuresCol='features', labelCol='GradeNumeric')
nb_model = nb.fit(train_data)

In [32]:
# Make predictions using the Naïve Bayes model
nb_predictions = nb_model.transform(test_data)
nb_predictions.toPandas()

,features,GradeNumeric,rawPrediction,probability,prediction
0,"[18.0, 0.0, 7.0, 10.0, 3.0, 0.0, 1.0]",1.0,"[-56.989172379676205, -58.18825756870697]","[0.7683620037379473, 0.23163799626205264]",0.0
1,"[18.0, 0.0, 18.0, 5.0, 0.0, 1.0, 3.0]",0.0,"[-59.832449147051165, -61.28061858945828]","[0.8097165495115405, 0.19028345048845954]",0.0
2,"[18.0, 0.0, 19.0, 2.0, 0.0, 0.0, 2.0]",0.0,"[-47.50963424847369, -48.00645421811208]","[0.6217117217546321, 0.3782882782453679]",0.0
3,"[18.0, 1.0, 6.0, 6.0, 4.0, 1.0, 1.0]",0.0,"[-59.10880890642384, -59.177862862721824]","[0.517256632338648, 0.48274336766135195]",0.0
4,"[18.0, 1.0, 6.0, 8.0, 4.0, 2.0, 3.0]",0.0,"[-73.80224859816722, -74.50772724675696]","[0.6694013378232683, 0.33059866217673156]",0.0
...,...,...,...,...,...
199,"[30.0, 0.0, 21.0, 2.0, 2.0, 0.0, 2.0]",0.0,"[-65.6521713804533, -64.6935327119038]","[0.27715083888341857, 0.7228491611165814]",1.0
200,"[30.0, 1.0, 16.0, 6.0, 2.0, 0.0, 4.0]",1.0,"[-79.61732085388662, -78.78533439938734]","[0.30322520872307585, 0.6967747912769242]",1.0
201,"[30.0, 1.0, 17.0, 7.0, 3.0, 1.0, 0.0]",0.0,"[-76.69074263855231, -77.12705184955617]","[0.6073792390780367, 0.3926207609219633]",0.0
202,"[30.0, 1.0, 17.0, 10.0, 1.0, 0.0, 0.0]",1.0,"[-72.67874350888253, -73.8126325850409]","[0.7565559000204704, 0.24344409997952968]",0.0


In [33]:
# Evaluate Naïve Bayes using Area Under ROC
nb_evaluator = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction', labelCol='GradeNumeric')
nb_area_under_roc = nb_evaluator.evaluate(nb_predictions)
print('Naïve Bayes Area Under ROC:', nb_area_under_roc)

Naïve Bayes Area Under ROC: 0.772790883646542


## 14. Missing values: simple replacement method

In the original code, we used `dropna()` to delete rows with missing values. This is easy, but it may remove too much data.

A common alternative is **imputation**, which means replacing missing values with reasonable values.

Simple examples:

- For a numeric column such as `StudyHours`, replace missing values with the **median**.
- For a categorical column such as `TypeOfSchool`, replace missing values with the **most common category**.

The dataset created in this notebook normally does not have missing values, so the code below is mainly a template for the discussion activity.

In [34]:
# Example 1: Replace missing numeric values with the median
median_study_hours = df.approxQuantile('StudyHours', [0.5], 0.01)[0]
df_imputed = df.na.fill({'StudyHours': median_study_hours})

print('Median StudyHours:', median_study_hours)

Median StudyHours: 15.0


In [35]:
# Example 2: Replace missing categorical values with the most common category
most_common_school = df.groupBy('TypeOfSchool').count().orderBy('count', ascending=False).first()[0]
df_imputed = df_imputed.na.fill({'TypeOfSchool': most_common_school})

print('Most common TypeOfSchool:', most_common_school)

Most common TypeOfSchool: Online


## 15. Discussion forum guidance

For **Learning Activity 1**, you can describe the decision tree in plain language. For example:

- What is the first/main split in the tree?
- Which features appear most important?
- Does the model seem to use study hours, participation, or school type?
- Does the model make intuitive sense?

For **Learning Activity 2**, explain how you would handle missing values in the most important attribute. A simple answer could be:

- If the important attribute is numeric, replace missing values with the median.
- If the important attribute is categorical, replace missing values with the most common category.
- If too many values are missing, the column may be unreliable, and the best solution may be to collect better data or avoid using that attribute.

## 16. Homework: investigate classification metrics

Please investigate the meaning of these metrics:

- accuracy;
- precision;
- recall;
- F1-score;
- Area Under ROC; and
- confusion matrix.

Try to explain each metric in one or two sentences using your own words.